In [14]:
import pandas as pd
import re
import time
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

In [15]:
def clean_fatal(val):
    val = str(val).strip().upper()
    if val == 'Y': return 1
    if val in ['N', 'N ', ' N']: return 0
    return None

In [16]:
def clean_age(val):
    val = str(val).lower()
    if 'teen' in val: return 15
    nums = re.findall(r'\d+', val)
    if nums: return int(nums[0])
    return None

In [17]:
def categorize_activity(val):
    val = str(val).lower()
    if 'surf' in val: return 'Surfing'
    if 'swim' in val or 'bath' in val: return 'Swimming'
    if 'fish' in val: return 'Fishing'
    if 'dive' in val or 'snorkel' in val: return 'Diving'
    if 'board' in val: return 'Board Sports'
    return 'Other/Unknown'

In [18]:
url = 'https://www.sharkattackfile.net/spreadsheets/GSAF5.xls'
print("Connecting to database and fetching records...")
df = pd.read_excel(url)
time.sleep(1)
print("\nData loaded successfully.")

Connecting to database and fetching records...

Data loaded successfully.


In [19]:
df['Age_Clean'] = df['Age'].apply(clean_age)
initial_count = len(df)
df = df.dropna(subset=['Age_Clean']).copy()
age_drop_count = initial_count - len(df)

# 2. General Column Cleaning
df['Is_Fatal'] = df['Fatal Y/N'].apply(clean_fatal)
df['Country'] = df['Country'].str.strip().str.upper().fillna('UNKNOWN')
df['State_Clean'] = df['State'].str.strip().str.title().fillna('Unknown')

df['Type_Clean'] = df['Type'].str.strip().str.replace('Boat', 'Boating').fillna('Questionable')

pre_type_filter_count = len(df)
df = df[df['Type_Clean'] != '?'].copy()
type_filter_removed = pre_type_filter_count - len(df)

df['Activity_Clean'] = df['Activity'].apply(categorize_activity)

df['Year'] = pd.to_numeric(df['Year'], errors='coerce')
df = df[(df['Year'] >= 1900) & (df['Year'] <= 2026)].copy()

cols_to_drop = [
    'Unnamed: 22', 'Unnamed: 21', 'original order', 'Case Number',
    'Case Number.1', 'href', 'href formula', 'pdf'
]

df = df.drop(columns=cols_to_drop, errors='ignore')

pre_dup_count = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = pre_dup_count - len(df)

print(f"--- Data Cleaning Report ---")
print(f"- Duplicate records removed: {duplicates_removed}")
print(f"- Final record count: {len(df)}")

df

--- Data Cleaning Report ---
- Duplicate records removed: 0
- Final record count: 3949


,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source,Age_Clean,Is_Fatal,State_Clean,Type_Clean,Activity_Clean
0,18th March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,Unknown,M,39,Injuries to both legs and feet,N,1715hrs,Great White Shark,US Sun: Mendocino Coast News:Melissa Michaelson:,39.0,0.0,California,Unprovoked,Surfing
3,5th March,2026.0,Unprovoked,AUSTRALIA,Queensland,Lady Elliott Island,Snorkeling,Unknown,M,50's,Lacerations elbow and abdomen,N,0815hrs,Unknown,News.com.au: ABC News: Andy Currie,50.0,0.0,Queensland,Unprovoked,Diving
4,22nd February,2026.0,Unprovoked,NEW CALEDONIA,Noumea,Anse Vata near Point Magnin,Wing Foiling,Cyril Chevalier,M,55,Deep wounds arms and legs,Y,?,Tiger or bull shark implicated,Johannes Marchand: Andy Currie,55.0,1.0,Noumea,Unprovoked,Other/Unknown
6,29th January,2026.0,Unprovoked,BRAZIL,Recife,Del Chifre Beach in Olinda,Swimming,Deivson Rocha Dantas,M,13,Right thigh and lower leg stripped of flesh,Y,?,Unknown bull and tiger sharks frequent the area,Kevin McMurray Trackingsharks.com: TV Globo: P...,13.0,1.0,Recife,Unprovoked,Swimming
9,20th January,2026.0,Unprovoked,AUSTRALIA,NSW,Point Plomber North of Port Macquarie,Surfing,Paul Zvirdinas,M,39,Minor cuts and abrasions,N,0830hrs,Bull shark,Bob Myatt GSAF,39.0,0.0,Nsw,Unprovoked,Surfing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6310,01-Dec-1901,1901.0,Unprovoked,AUSTRALIA,Queensland,Brisbane,Bathing,William Quince,M,10,Lacerations to torso & thigh,N,NaN,NaN,"The Argus, 12/2/1901",10.0,0.0,Queensland,Unprovoked,Swimming
6312,Reported 23-Sep-1901,1901.0,Unprovoked,CYPRUS,Southern Cyprus,Larnaca,Swimming,male,M,Teen,"FATAL, bitten on arms, chest & legs",Y,NaN,2 m shark,"Bardanis citing Embros, 9/23/1901",15.0,1.0,Southern Cyprus,Unprovoked,Swimming
6313,30-Jul-1901,1901.0,Unprovoked,SOUTH AFRICA,Western Cape Province,Windmill Beach,Swimming,"John Hendrick Adrian Chandler, a prisoner of war",M,29,"Right leg bitten & foot severed, right arm bit...",Y,14h15,White shark,"M. Levine, GSAF",29.0,1.0,Western Cape Province,Unprovoked,Swimming
6314,17-Jul-1901,1901.0,Invalid,ITALY,Syracuse,Capo Santa Croce,Swimming,Antonio Tornatori,M,20,"Disappeared, but shark involvement unconfirmed",NaN,NaN,Questionable,C. Moore. GSAF,20.0,NaN,Syracuse,Invalid,Swimming


In [7]:
print("\n[TABLE 1] TOP 10 HIGH-INCIDENT COUNTRIES")
country_counts = df['Country'].value_counts().head(10).reset_index()
country_counts.columns = ['Country', 'Incident Count']
country_counts


[TABLE 1] TOP 10 HIGH-INCIDENT COUNTRIES


,Country,Incident Count
0,USA,1745
1,AUSTRALIA,839
2,SOUTH AFRICA,375
3,BAHAMAS,91
4,BRAZIL,74
5,NEW ZEALAND,60
6,MEXICO,56
7,PAPUA NEW GUINEA,47
8,REUNION,35
9,NEW CALEDONIA,33


In [8]:
print("\n[TABLE 2] TOP 10 HIGH-INCIDENT STATES/AREAS")
state_counts = df['State_Clean'].value_counts().head(10).reset_index()
state_counts.columns = ['State/Area', 'Incident Count']
state_counts


[TABLE 2] TOP 10 HIGH-INCIDENT STATES/AREAS


,State/Area,Incident Count
0,Florida,937
1,New South Wales,298
2,California,219
3,Queensland,218
4,Hawaii,176
5,Kwazulu-Natal,140
6,Unknown,134
7,Western Australia,129
8,Eastern Cape Province,116
9,Western Cape Province,111


In [9]:
print("\n[TABLE 3] ACTIVITY RISK PROFILE (CATEGORIZED)")
activity_analysis = df.groupby('Activity_Clean')['Is_Fatal'].agg(['count', 'mean'])
activity_analysis = activity_analysis.sort_values(by='count', ascending=False).reset_index()
activity_analysis.columns = ['Category', 'Volume', 'Fatality Rate']
activity_analysis['Fatality Rate'] = activity_analysis['Fatality Rate'].map(
    lambda x: f"{x:.2%}" if pd.notnull(x) else "0.00%")
activity_analysis


[TABLE 3] ACTIVITY RISK PROFILE (CATEGORIZED)


,Category,Volume,Fatality Rate
0,Surfing,1108,7.58%
1,Other/Unknown,956,19.25%
2,Swimming,788,32.49%
3,Fishing,566,13.78%
4,Board Sports,167,17.37%
5,Diving,115,21.74%


In [10]:
print("\n[TABLE 4] INCIDENT TYPE ANALYSIS")
type_analysis = df['Type_Clean'].value_counts().reset_index()
type_analysis.columns = ['Incident Type', 'Count']
type_analysis


[TABLE 4] INCIDENT TYPE ANALYSIS


,Incident Type,Count
0,Unprovoked,3271
1,Provoked,315
2,Invalid,236
3,Watercraft,53
4,Sea Disaster,43
5,Questionable,29
6,unprovoked,1
7,Under investigation,1



**H4 — Overall Incident profile**

**Findings:**

The cleaned dataset shows a clear concentration of incidents in a few places and activities.

- The **USA**, **Australia**, and **South Africa** account for the largest number of incidents.
- **Florida**, **New South Wales**, and **California** are the highest-incident regions.
- **Surfing** is the most common activity in the data, but it has a relatively low fatality rate compared with several other activities.
- **Swimming** and **Other/Unknown** activities show higher fatality rates, which suggests that risk is not driven only by incident volume.
- Most incidents are classified as **Unprovoked**, meaning the dataset is dominated by events that happened during normal ocean use rather than deliberate human interaction.

**Conclusions:**

Overall, the analysis suggests that shark risk is not evenly distributed.
It is concentrated by **location**, **activity**, and **incident type**.

The most important takeaway is that:
- high-incident areas are not always the most dangerous in severity terms,
- some lower-volume categories have much worse fatality outcomes,
- and unprovoked incidents remain the most important category for underwriting and claims severity.

**Recommendations for insurers:**

- Focus pricing and reserves on **high-risk geographies** rather than using one global risk assumption.
- Treat **activity type** as a key severity factor, especially for swimming and unknown activity cases.
- Use **unprovoked incident history** as the main indicator of severe-loss exposure.
- Avoid relying on incident counts alone, since volume does not always match fatality risk.
- Combine location, activity, and incident type into a broader risk model for more accurate pricing.